# LAB 05 - Bài 26: Sentiment Classifier Application

Pipeline: raw review → tokenizer → token IDs → padding/truncation → Embedding → LSTM → sentiment probability.



In [1]:
# ============================================================
# BÀI 26 - APPLICATION: HUẤN LUYỆN SENTIMENT CLASSIFIER
# Pipeline: raw review -> tokenizer -> token IDs -> padding
#           -> Embedding -> LSTM -> sentiment probability.
# ============================================================

from pathlib import Path
import tarfile
import urllib.request
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models

DATA_URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
DATA_DIR = Path("/content/aclImdb")
ARCHIVE_PATH = Path("/content/aclImdb_v1.tar.gz")

# Tải và giải nén dataset nếu chưa có trong Colab.
if not (DATA_DIR / "train" / "pos").exists():
    if not ARCHIVE_PATH.exists():
        print("Đang tải ACL IMDb từ Stanford (~84 MB)...")
        urllib.request.urlretrieve(DATA_URL, ARCHIVE_PATH)
    print("Đang giải nén dataset...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as archive:
        archive.extractall(path="/content")

# Dùng train split để huấn luyện và tạo validation; nhãn neg=0, pos=1.
train_texts_all, labels_all = [], []
for label_name, label_id in (("neg", 0), ("pos", 1)):
    folder = DATA_DIR / "train" / label_name
    for file_path in sorted(folder.glob("*.txt")):
        train_texts_all.append(file_path.read_text(encoding="utf-8", errors="replace").strip())
        labels_all.append(label_id)
labels_all = np.asarray(labels_all, dtype=np.int32)

train_texts, val_texts, y_train, y_val = train_test_split(
    train_texts_all,
    labels_all,
    test_size=0.1,
    random_state=42,
    stratify=labels_all,
)

MAX_VOCAB_SIZE = 20_000
MAX_LEN = 200
EMBEDDING_DIM = 64
HIDDEN_SIZE = 64
BATCH_SIZE = 64
EPOCHS = 3
SEED = 42

# Vocabulary chỉ được fit trên train data.
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)
vocab_size = min(MAX_VOCAB_SIZE, len(tokenizer.word_index) + 1)

X_train = pad_sequences(
    tokenizer.texts_to_sequences(train_texts),
    maxlen=MAX_LEN, padding="post", truncating="post", value=0
)
X_val = pad_sequences(
    tokenizer.texts_to_sequences(val_texts),
    maxlen=MAX_LEN, padding="post", truncating="post", value=0
)

# Embedding -> LSTM -> output probability for positive class.
tf.keras.utils.set_random_seed(SEED)
model = models.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(vocab_size, EMBEDDING_DIM, mask_zero=True),
    layers.LSTM(HIDDEN_SIZE),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    verbose=2,
)

MODEL_PATH = "/content/lab05_sentiment_lstm.keras"
model.save(MODEL_PATH)
print("Saved model:", MODEL_PATH)


Đang tải ACL IMDb từ Stanford (~84 MB)...
Đang giải nén dataset...


/tmp/ipykernel_552/179626642.py:28: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(path="/content")


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 64)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,089 (5.01 MB)

 Trainable params: 1,313,089 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
352/352 - 41s - 116ms/step - accuracy: 0.7555 - loss: 0.5006 - val_accuracy: 0.8316 - val_loss: 0.4183
Epoch 2/3
352/352 - 40s - 113ms/step - accuracy: 0.8973 - loss: 0.2636 - val_accuracy: 0.8444 - val_loss: 0.3803
Epoch 3/3
352/352 - 42s - 120ms/step - accuracy: 0.9270 - loss: 0.2037 - val_accuracy: 0.7600 - val_loss: 0.5090
Saved model: /content/lab05_sentiment_lstm.keras


In [2]:
# ============================================================
# BÀI 26 - DỰ ĐOÁN REVIEW NGẮN / DÀI / PHỦ ĐỊNH / TƯƠNG PHẢN
# ============================================================

import pandas as pd

reviews = [
    {
        "type": "Short review",
        "text": "I loved this movie.",
        "analysis_hint": "Từ 'loved' là tín hiệu sentiment tích cực rõ ràng.",
    },
    {
        "type": "Long review",
        "text": (
            "The film takes its time introducing the characters and the small town where they live. "
            "The cinematography is thoughtful, the dialogue feels natural, and the actors give "
            "believable performances. A few scenes move slowly, but the story remains engaging "
            "and the ending is satisfying. I enjoyed the film and would recommend it to friends."
        ),
        "analysis_hint": "Cân nhắc các đoạn 'engaging', 'satisfying', 'enjoyed' cùng chi tiết chê nhẹ về nhịp phim.",
    },
    {
        "type": "Negation",
        "text": "I did not enjoy this movie; the plot was dull and the acting felt lifeless.",
        "analysis_hint": "Xem model kết hợp 'not' với 'enjoy' ra sao; 'dull' và 'lifeless' cũng nghiêng tiêu cực.",
    },
    {
        "type": "Contrast",
        "text": "I expected the film to be terrible, but the performances were excellent and I enjoyed it.",
        "analysis_hint": "Chú ý 'terrible' ở vế đầu đối lập với 'but', 'excellent' và 'enjoyed' ở vế sau.",
    },
]

texts = [item["text"] for item in reviews]
sequences = tokenizer.texts_to_sequences(texts)
X_examples = pad_sequences(
    sequences, maxlen=MAX_LEN, padding="post", truncating="post", value=0
)
probabilities = model.predict(X_examples, verbose=0).ravel()

rows = []
for item, sequence, probability in zip(reviews, sequences, probabilities):
    rows.append({
        "type": item["type"],
        "true/expected label": "(tự đánh giá sau khi xem kết quả)",
        "prediction": "Positive" if probability >= 0.5 else "Negative",
        "P(Positive)": float(probability),
        "tokens before padding": len(sequence),
        "analysis hint": item["analysis_hint"],
        "truncated": len(sequence) > MAX_LEN,
    })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False, formatters={"P(Positive)": "{:.4f}".format}))

# Lưu kết quả dự đoán để đưa vào báo cáo.
OUTPUT_CSV = "/content/lab05_bai26_predictions.csv"
results_df.to_csv(OUTPUT_CSV, index=False)
print("\nĐã lưu kết quả:", OUTPUT_CSV)

# 'analysis_hint' là gợi ý để người học phân tích, không phải giải thích
# được trích xuất tự động từ model (model này không có attention/explainer).


        type               true/expected label prediction P(Positive)  tokens before padding                                                                             analysis hint  truncated
Short review (tự đánh giá sau khi xem kết quả)   Positive      0.7280                      4                                        Từ 'loved' là tín hiệu sentiment tích cực rõ ràng.      False
 Long review (tự đánh giá sau khi xem kết quả)   Positive      0.9620                     54 Cân nhắc các đoạn 'engaging', 'satisfying', 'enjoyed' cùng chi tiết chê nhẹ về nhịp phim.      False
    Negation (tự đánh giá sau khi xem kết quả)   Negative      0.1331                     15   Xem model kết hợp 'not' với 'enjoy' ra sao; 'dull' và 'lifeless' cũng nghiêng tiêu cực.      False
    Contrast (tự đánh giá sau khi xem kết quả)   Positive      0.9171                     16           Chú ý 'terrible' ở vế đầu đối lập với 'but', 'excellent' và 'enjoyed' ở vế sau.      False

Đã lưu kết quả: /content/lab0